# Benchmarks: can anyone see the turn?

Notebook 03 ended with a puzzle: in flu season the model **under-forecasts the climb and overshoots when forecasting from the top of the curve**, and that is where almost all the 4-week error, and the staffing cost, comes from.

This notebook asks whether the turn of the season can be seen **four weeks ahead** from any available signal, and how the professionals do:
1. The national curve's own shape (does growth slow before a peak?)
2. States peaking before the nation
3. Outpatient flu-like illness (ILINet)
4. CDC's FluSight ensemble and baseline: accuracy, ranges and staffing cost
5. A mini-ensemble of our own

## 0. Setup

Same data, model, floor and 97 forecast dates as notebook 03.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import io, json, urllib.request
import pandas as pd, numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

HUB = 'https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/main/'
raw = pd.read_csv(HUB + 'target-data/target-hospital-admissions.csv', parse_dates=['date'])
y = raw[raw.location == 'US'].set_index('date')['value'].sort_index().asfreq('W-SAT')

cutoffs = pd.date_range('2024-07-27', '2026-05-30', freq='W-SAT')
WINTER = [11, 12, 1, 2, 3, 4]
FULL_SEASONS = [2022, 2023, 2024, 2025]          # complete Jul-Jun seasons in the data
season_of = lambda d: np.where(d.month >= 7, d.year, d.year - 1)

def fit_sarima(series):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return SARIMAX(series, order=(1,1,1), seasonal_order=(1,0,0,52),
                       enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

rows = []
for c in cutoffs:
    f = fit_sarima(y[:c]).get_forecast(4)
    ci = f.conf_int(alpha=0.2)
    floor = y[:c].min()                          # never below the lowest week seen so far
    for h in range(4):
        week = f.predicted_mean.index[h]
        rows.append({'cutoff': c, 'horizon': h + 1, 'target_week': week,
                     'mean': max(f.predicted_mean.iloc[h], floor),
                     'q10': ci.iloc[h, 0], 'q90': ci.iloc[h, 1], 'actual': y[week]})
bt = pd.DataFrame(rows)
bt['winter'] = bt['target_week'].dt.month.isin(WINTER)
bt['ratio']  = bt['actual'] / bt['mean']
print(len(bt), 'forecasts')

388 forecasts


## 1. Does the curve's own shape warn of the turn?

Phase must be known on the forecast date. Using this week's change (`g1`) and last week's (`g0`):
- **accelerating**: rising at least as fast as last week
- **decelerating**: still rising, but slower: the classic sign of an approaching peak
- **falling**

In [2]:
y0 = y[bt['cutoff']].values
y1 = y[bt['cutoff'] - pd.Timedelta(weeks=1)].values
y2 = y[bt['cutoff'] - pd.Timedelta(weeks=2)].values
bt['g1'], bt['g0'] = y0 - y1, y1 - y2
bt['phase'] = np.select([bt['g1'] <= 0, bt['g1'] >= bt['g0']], ['falling', 'accelerating'], default='decelerating')

w4 = bt[(bt['horizon'] == 4) & bt['winter']]
print('Flu-season 4-week forecasts per phase:'); print(w4.groupby('phase').size())
print('\nactual / forecast, 10th / 50th / 90th percentile:')
print(w4.groupby('phase')['ratio'].quantile([0.10, 0.50, 0.90]).unstack().round(2))

Flu-season 4-week forecasts per phase:
phase
accelerating    25
decelerating     7
falling         20
dtype: int64

actual / forecast, 10th / 50th / 90th percentile:
               0.1   0.5   0.9
phase                         
accelerating  0.58  1.68  2.94
decelerating  0.40  0.97  2.98
falling       0.39  0.97  6.44


In [3]:
# The big overshoots: when were they made, and what was growth doing?
w4[(w4['phase'] == 'accelerating') & (w4['ratio'] < 0.8)][['cutoff', 'target_week', 'g0', 'g1', 'mean', 'actual', 'ratio']].round(2)

,cutoff,target_week,g0,g1,mean,actual,ratio
91,2024-12-28,2025-01-25,6527.0,12525.0,63777.40,41400.0,0.65
111,2025-02-01,2025-03-01,7724.0,11120.0,61576.98,31906.0,0.52
295,2025-12-20,2026-01-17,3687.0,9920.0,36964.17,19735.0,0.53
299,2025-12-27,2026-01-24,9920.0,16448.0,54720.57,16819.0,0.31


**What it shows.**
- **Accelerating:** the median forecast is far too low (reality ≈ 1.7× the forecast). Momentum decays faster than real epidemic growth.
- **Decelerating:** only 7 examples, median on target. It did **not** isolate the overshoots.
- **The overshoots** were all made **1-2 weeks before a peak** (4 Jan 2025, 8 Feb 2025, 3 Jan 2026), with targets after it, and in every case growth was still **accelerating**. These peaks arrived at full speed: there was no slowdown to detect.

## 2. Do some states peak before the nation?

If some states reliably turn first, their turn is a warning the national curve can't give. Weeks each state peaked **before** the national peak (positive = earlier), complete seasons only.

In [4]:
st = raw[raw['location'] != 'US'].copy()
st['season'] = season_of(st['date'].dt)
st = st[st['season'].isin(FULL_SEASONS)]
peaks = st.loc[st.groupby(['season', 'location_name'])['value'].idxmax(), ['season', 'location_name', 'date']]

us = raw[raw['location'] == 'US'].copy()
us['season'] = season_of(us['date'].dt)
us_peak = us.loc[us.groupby('season')['value'].idxmax()].set_index('season')['date']

peaks['weeks_early'] = (peaks['season'].map(us_peak) - peaks['date']).dt.days // 7
table = peaks.pivot(index='location_name', columns='season', values='weeks_early')
print('National peaks:', us_peak[FULL_SEASONS].dt.date.to_dict(), '\n')
print(table.loc[table.mean(axis=1).sort_values(ascending=False).index].head(12))

National peaks: {2022: datetime.date(2022, 12, 3), 2023: datetime.date(2023, 12, 30), 2024: datetime.date(2025, 2, 8), 2025: datetime.date(2026, 1, 3)} 

season          2022  2023  2024  2025
location_name                         
Alaska            -1     7     5     1
Puerto Rico       -3     9     4     0
Nevada             0     0     5     1
Louisiana          0     0     5     1
Minnesota          0     0     5     0
New Mexico        -1     0     5     0
Wyoming            0    -1     5     0
South Carolina     3     0     1     0
Idaho             -2     0     6     0
Mississippi        1     0     1     1
Georgia            0     0     1     1
Colorado          -3     0     5     0


**What it shows.**
- In 2022-23, 2023-24 and 2025-26 nearly every state peaked in the **same week as the nation**, or one week off.
- 2024-25 shows many states "5 weeks early", but that is the **double peak**: their highest week was the first hump (4 Jan), not an early warning.
- No state is early in every season. US flu peaks are nearly simultaneous nationwide (two of four landed at New Year).
- *Limit:* this compares peak weeks only, over four seasons.

## 3. Does outpatient flu-like illness (ILINet) peak earlier?

People see a doctor before they're hospitalized, so ILINet (weighted % of outpatient visits that are flu-like) might lead admissions. Source: the Delphi Epidata API (Carnegie Mellon), which serves CDC's ILINet.

*This cell needs access to `api.delphi.cmu.edu`. The results below were recorded from a run in October 2026.*

In [ ]:
url = 'https://api.delphi.cmu.edu/epidata/fluview/?regions=nat&epiweeks=202140-202640'

def week_end(ew):
    """MMWR epiweek (e.g. 202503) -> the Saturday ending it, same labelling as admissions."""
    year, wk = divmod(int(ew), 100)
    jan4 = pd.Timestamp(year, 1, 4)
    return jan4 + pd.Timedelta(days=(5 - jan4.weekday()) % 7) + pd.Timedelta(weeks=wk - 1)

try:
    ili_raw = pd.DataFrame(json.load(urllib.request.urlopen(url, timeout=30))['epidata'])
    ili_raw['date'] = ili_raw['epiweek'].map(week_end)
    ili = ili_raw.set_index('date')['wili'].sort_index()
    both = pd.DataFrame({'admissions': y, 'ili': ili}).dropna()
    both['season'] = season_of(both.index)
    pk = (both[both['season'].isin(FULL_SEASONS)].groupby('season')
              .agg(adm_peak=('admissions', 'idxmax'), ili_peak=('ili', 'idxmax')))
    pk['ili_weeks_earlier'] = (pk['adm_peak'] - pk['ili_peak']).dt.days // 7
    print(pk)
except Exception as e:
    print('ILINet not reachable from here:', type(e).__name__)

**Result (October 2026 run):**

| Season | Admissions peak | ILI peak | ILI earlier by |
|---|---|---|---|
| 2022-23 | 3 Dec 2022 | 26 Nov 2022 | 1 week |
| 2023-24 | 30 Dec 2023 | 30 Dec 2023 | 0 |
| 2024-25 | 8 Feb 2025 | 8 Feb 2025 | 0 |
| 2025-26 | 3 Jan 2026 | 27 Dec 2025 | 1 week |

To help a 4-week forecast see the turn, ILI would need to peak 3-4 weeks earlier, consistently. It peaks **0-1 weeks** earlier. At most it could sharpen 1-week forecasts.

## 4. The professionals: CDC FluSight

Each flu season, dozens of teams submit 4-week forecasts to CDC's FluSight hub.
- **FluSight-ensemble**: for each percentile, the **median across all submitted teams**.
- **FluSight-baseline**: the latest week carried forward flat, with a range from past week-to-week changes.

Both publish 23 percentiles per forecast. Their `reference_date − 1 week` = our `cutoff`; their `horizon + 1` = our horizon.

**Fairness caveat:** FluSight teams forecast in **real time** from incomplete data that was later revised; our backtest uses the **final** data. The comparison is tilted in our model's favour.

In [6]:
BASE = HUB + 'model-output/{m}/{d}-{m}.csv'
parts = []
for m in ['FluSight-ensemble', 'FluSight-baseline']:
    for r in pd.date_range('2024-08-03', '2026-06-06', freq='W-SAT'):
        try:
            data = urllib.request.urlopen(BASE.format(m=m, d=r.strftime('%Y-%m-%d'))).read()
        except Exception:
            continue                                  # no submission that week (off-season)
        f = pd.read_csv(io.BytesIO(data), dtype={'location': str, 'output_type_id': str})
        f = f[(f['location'] == 'US') & (f['target'] == 'wk inc flu hosp') & (f['output_type'] == 'quantile')]
        f['model'] = m
        parts.append(f)

fs = pd.concat(parts)
fs['cutoff']  = pd.to_datetime(fs['reference_date']) - pd.Timedelta(weeks=1)
fs['horizon'] = fs['horizon'] + 1
fs = fs[fs['horizon'].between(1, 4)]
print(fs.groupby('model')['cutoff'].agg(['nunique', 'min', 'max']))

                   nunique        min        max
model                                           
FluSight-baseline       55 2024-11-16 2026-05-23
FluSight-ensemble       55 2024-11-16 2026-05-23


### 4a. Accuracy of the middle line

In [7]:
med = (fs[fs['output_type_id'] == '0.5']
         .pivot_table(index=['cutoff', 'horizon'], columns='model', values='value').reset_index())
m = med.merge(bt[['cutoff', 'horizon', 'mean', 'actual']], on=['cutoff', 'horizon'])
print(m['cutoff'].nunique(), 'forecast dates compared (flu season only)\n')
for col in ['FluSight-ensemble', 'FluSight-baseline', 'mean']:
    m['ae_' + col] = (m[col] - m['actual']).abs()
print('MAE by weeks ahead:')
print(m.groupby('horizon')[['ae_FluSight-ensemble', 'ae_FluSight-baseline', 'ae_mean']].mean().round(0))

55 forecast dates compared (flu season only)

MAE by weeks ahead:
         ae_FluSight-ensemble  ae_FluSight-baseline  ae_mean
horizon                                                     
1.0                    2652.0                3328.0   2550.0
2.0                    4144.0                5750.0   5716.0
3.0                    5638.0                8104.0   8099.0
4.0                    7104.0               10024.0   9765.0


In [8]:
# The four forecasts where our model overshot most: did the professionals see the turn?
turns = pd.to_datetime(['2024-12-28', '2025-02-01', '2025-12-20', '2025-12-27'])
m[(m['horizon'] == 4) & m['cutoff'].isin(turns)][['cutoff', 'actual', 'mean', 'FluSight-ensemble', 'FluSight-baseline']].round(0)

,cutoff,actual,mean,FluSight-ensemble,FluSight-baseline
27,2024-12-28,41400.0,63777.0,23735.0,25693.0
43,2025-02-01,31906.0,61577.0,33810.0,47414.0
131,2025-12-20,19735.0,36964.0,30159.0,18546.0
135,2025-12-27,16819.0,54721.0,39965.0,33494.0


**What it shows.**
- At 2-4 weeks the ensemble is clearly better: at 4 weeks its error is **~27% lower** than ours. At 1 week it's a tie.
- Our SARIMA performs almost exactly like CDC's own **baseline**.
- At the turns, the ensemble **nailed one peak** (1 Feb 2025, within 6%), **overshot one by 138%** (27 Dec 2025) and **undershot one** (28 Dec 2024). The flat baseline did well at two peaks: flat is a good bet when things are about to stop rising.

### 4b. Are their ranges honest?

In [9]:
rng = (fs[fs['output_type_id'].isin(['0.1', '0.9'])]
         .pivot_table(index=['cutoff', 'horizon'], columns=['model', 'output_type_id'], values='value'))
rng.columns = [f'{mm}_{q}' for mm, q in rng.columns]
r = rng.reset_index().merge(bt[['cutoff', 'horizon', 'q10', 'q90', 'actual']], on=['cutoff', 'horizon'])

for name, lo, hi in [('ensemble', 'FluSight-ensemble_0.1', 'FluSight-ensemble_0.9'),
                     ('baseline', 'FluSight-baseline_0.1', 'FluSight-baseline_0.9'),
                     ('our SARIMA', 'q10', 'q90')]:
    inside = ((r['actual'] >= r[lo]) & (r['actual'] <= r[hi])).mean()
    above, below = (r['actual'] > r[hi]).mean(), (r['actual'] < r[lo]).mean()
    print(f'{name:11} inside {inside:4.0%}   above q90 {above:4.0%}   below q10 {below:4.0%}   (targets 80 / 10 / 10)')

ensemble    inside  77%   above q90  20%   below q10   3%   (targets 80 / 10 / 10)
baseline    inside  60%   above q90  24%   below q10  17%   (targets 80 / 10 / 10)
our SARIMA  inside  61%   above q90  20%   below q10  20%   (targets 80 / 10 / 10)


**What it shows.**
- The ensemble's 80% range covers ~77%, but its misses are almost all **above**: reality beat its q90 ~20% of the time and almost never fell below q10. Its range sits too low.
- On the staffing-critical upper edge, our SARIMA ties it (~20%). The ensemble's better coverage comes from its lower edge.
- A plausible reason the professionals lean low: they forecast from real-time reports, which are often revised **upward** later. The vintage archive will be able to test this.

### 4c. Staffing from each forecast

Same rule as notebook 03: capacity set 4 weeks ahead; understaffing costs 3 per admission, overstaffing 1 (cost points, assumed ratio).

In [10]:
def cost(actual, capacity, ratio=3):
    return np.maximum(capacity - actual, 0).sum() + ratio * np.maximum(actual - capacity, 0).sum()

p = (fs[(fs['horizon'] == 4) & fs['output_type_id'].isin(['0.5', '0.75', '0.9'])]
       .pivot_table(index='cutoff', columns=['model', 'output_type_id'], values='value'))
p.columns = [f'{mm}_{q}' for mm, q in p.columns]
s = p.reset_index().merge(bt[bt['horizon'] == 4][['cutoff', 'actual', 'mean', 'q90']], on='cutoff')
print(len(s), 'forecast dates\n')

plans = {'ensemble median': 'FluSight-ensemble_0.5', 'ensemble q75': 'FluSight-ensemble_0.75',
         'ensemble q90': 'FluSight-ensemble_0.9', 'baseline (flat)': 'FluSight-baseline_0.5',
         'our SARIMA mean': 'mean', 'our SARIMA q90': 'q90'}
for name, col in plans.items():
    short = (s['actual'] > s[col]).sum()
    print(f"{name:16} cost {cost(s['actual'], s[col]):>11,.0f}   weeks short {short:>2} ({short / len(s):.0%})")

55 forecast dates

ensemble median  cost     978,074   weeks short 35 (64%)
ensemble q75     cost     842,135   weeks short 17 (31%)
ensemble q90     cost     807,251   weeks short 13 (24%)
baseline (flat)  cost   1,153,144   weeks short 17 (31%)
our SARIMA mean  cost     997,584   weeks short 22 (40%)
our SARIMA q90   cost   1,083,975   weeks short 11 (20%)


**What it shows.**
- With the ensemble, staffing **above** the middle line pays: median → q75 → q90 gets cheaper. With our SARIMA it was the reverse.
- The ensemble's "q90" was short ~24% of the time, so it behaves like an honest **~q75**, and it won. **The newsvendor rule works: the best plan is the one that's really short ~25% of the time, whatever its label.**
- The professionals' best plan is **~19% cheaper** than ours. Our middle line is about as good as theirs for staffing; what they have is a range trustworthy enough to staff above.

## 5. A mini-ensemble of our own

Average two models with opposite strengths: SARIMA (follows momentum, good on climbs) and flat (latest week carried forward, good at peaks).

In [11]:
bt['flat'] = y[bt['cutoff']].values
bt['combo'] = (bt['mean'] + bt['flat']) / 2
m2 = m.merge(bt[['cutoff', 'horizon', 'flat', 'combo']], on=['cutoff', 'horizon'])
for col in ['flat', 'combo']:
    m2['ae_' + col] = (m2[col] - m2['actual']).abs()
print(m2.groupby('horizon')[['ae_FluSight-ensemble', 'ae_mean', 'ae_flat', 'ae_combo']].mean().round(0))

         ae_FluSight-ensemble  ae_mean  ae_flat  ae_combo
horizon                                                  
1.0                    2652.0   2550.0   3593.0    2877.0
2.0                    4144.0   5716.0   6422.0    5890.0
3.0                    5638.0   8099.0   8731.0    7937.0
4.0                    7104.0   9765.0  10850.0    9464.0


**What it shows.** The combo beats SARIMA at 3-4 weeks by only ~3%, and is worse at 1-2. Both members look at exactly the same information (the national series), so they're wrong in the same weeks and averaging can't cancel much. The professional ensemble gains because its members see **different data**. **The diversity that matters is in the data, not the method.**

## 6. Summary

| Check | Result |
|---|---|
| Curve shape (slowing growth) | Peaks arrived at full speed; the big overshoots were made 1-2 weeks before peaks while growth was still accelerating |
| States vs nation | Same week or ±1; no consistently early states |
| Outpatient ILI | Peaks 0-1 weeks earlier, not the 3-4 needed |
| FluSight ensemble accuracy | ~27% lower 4-week error than ours; still overshot one peak by 138% |
| Ranges | Everyone underestimates upside risk: ensemble and ours both beaten above q90 ~20% (promised 10%) |
| Staffing | Ensemble's best plan ~19% cheaper than ours; its q90 behaves like an honest q75 and wins, as the newsvendor rule predicts |
| Mini-ensemble | −3% at 4 weeks: averaging models that see the same data barely helps |

**In one paragraph.** At a four-week horizon, the turn of the flu season was not visible in the national curve, in state-level data, or in outpatient visits, and even CDC's professional ensemble missed it at one of the three recent peaks. What separates the professionals is less the middle line than the range: theirs is trustworthy enough to staff above, which made their best staffing plan about 19% cheaper than ours. The path to better forecasts here is new, diverse information (pooling across states, real-time data handling), not more tuning of a single national model.